In [1]:
1+1

2

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from agents import Agent, Runner, function_tool
from dotenv import load_dotenv
import os
import yt_dlp
from typing import Any
import requests


# Experimenting with FPL API

In [3]:
from __future__ import annotations

import requests
from typing import Any


class FPLService:
    BASE_URL = "https://fantasy.premierleague.com/api"

    def __init__(self, timeout: int = 20) -> None:
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "fpl-agent/0.1",
                "Accept": "application/json",
            }
        )

    def _get(self, path: str) -> Any:
        url = f"{self.BASE_URL}{path}"
        response = self.session.get(url, timeout=self.timeout)
        response.raise_for_status()
        return response.json()

    def get_bootstrap_static(self) -> dict[str, Any]:
        return self._get("/bootstrap-static/")

    def get_fixtures(self) -> list[dict[str, Any]]:
        return self._get("/fixtures/")

    def get_event_live(self, gameweek: int) -> dict[str, Any]:
        return self._get(f"/event/{gameweek}/live/")

    def get_element_summary(self, player_id: int) -> dict[str, Any]:
        return self._get(f"/element-summary/{player_id}/")

    def get_entry(self, team_id: int) -> dict[str, Any]:
        return self._get(f"/entry/{team_id}/")

    def get_entry_picks(self, team_id: int, gameweek: int) -> dict[str, Any]:
        return self._get(f"/entry/{team_id}/event/{gameweek}/picks/")

    def get_my_team(self, team_id: int, gameweek: int) -> dict[str, Any]:
        return self._get(f"/entry/{team_id}/event/{gameweek}/picks/")

    def get_player_news(self) -> list[dict[str, Any]]:
        data = self.get_bootstrap_static()
        players = data.get("elements", [])
        teams = {team["id"]: team["name"] for team in data.get("teams", [])}

        news_items: list[dict[str, Any]] = []

        for player in players:
            news_text = (player.get("news") or "").strip()
            if not news_text:
                continue

            news_items.append(
                {
                    "player_id": player["id"],
                    "name": player["web_name"],
                    "team_id": player["team"],
                    "team_name": teams.get(player["team"]),
                    "status": player.get("status"),
                    "chance_of_playing_next_round": player.get("chance_of_playing_next_round"),
                    "chance_of_playing_this_round": player.get("chance_of_playing_this_round"),
                    "news": news_text,
                }
            )

        return news_items

In [7]:
import pandas as pd

fpl = FPLService()
bootstrap = fpl.get_bootstrap_static()

players_df = pd.DataFrame(bootstrap["elements"])[
    [
        "id",
        "first_name",
        "second_name",
        "web_name",
        "team",
        "element_type",
        "now_cost",
        "total_points",
        "minutes",
        "goals_scored",
        "assists",
        "selected_by_percent",
    ]
]

teams_df = pd.DataFrame(bootstrap["teams"])[
    [
        "id",
        "name",
        "short_name",
        "strength",
        "strength_attack_home",
        "strength_attack_away",
        "strength_defence_home",
        "strength_defence_away",
    ]
]

gameweeks_df = pd.DataFrame(bootstrap["events"])[
    [
        "id",
        "name",
        "deadline_time",
        "finished",
        "is_current",
        "is_next",
    ]
]


In [40]:
players_df.head(50)


,id,first_name,second_name,web_name,team,element_type,now_cost,total_points,minutes,goals_scored,assists,selected_by_percent
0,1,David,Raya Martín,Raya,1,1,60,129,2790,0,0,34.6
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,1,1,40,0,0,0,0,0.4
2,3,Karl,Hein,Hein,1,1,40,0,0,0,0,0.2
3,4,Tommy,Setford,Setford,1,1,39,0,0,0,0,0.2
4,5,Gabriel,dos Santos Magalhães,Gabriel,1,2,72,173,2165,3,4,43.0
5,6,William,Saliba,Saliba,1,2,61,107,2074,1,0,13.2
6,7,Riccardo,Calafiori,Calafiori,1,2,56,93,1446,1,2,6.0
7,8,Jurriën,Timber,J.Timber,1,2,63,149,2452,3,6,26.8
8,9,Jakub,Kiwior,Kiwior,1,2,54,0,0,0,0,0.0
9,10,Myles,Lewis-Skelly,Lewis-Skelly,1,2,50,12,311,0,0,1.4


## Creating agent tools

In [ ]:
# src/tools/fpl_api.py
from __future__ import annotations

from typing import Any
from pydantic import BaseModel

from src.services.fpl_service import FPLService

fpl_service = FPLService()


class BootstrapResponse(BaseModel):
    """Structured summary of the main FPL bootstrap dataset."""

    total_players: int
    total_teams: int
    total_events: int
    sample_players: list[dict[str, Any]]


def get_bootstrap_data() -> dict[str, Any]:
    """Return a compact summary of the FPL bootstrap-static dataset.

    Includes overall counts for players, teams, and events, plus a small
    sample of player records for quick inspection.
    """
    data = fpl_service.get_bootstrap_static()
    return BootstrapResponse(
        total_players=len(data.get("elements", [])),
        total_teams=len(data.get("teams", [])),
        total_events=len(data.get("events", [])),
        sample_players=[
            {
                "id": p["id"],
                "web_name": p["web_name"],
                "team": p["team"],
                "now_cost": p["now_cost"],
                "total_points": p["total_points"],
                "form": p["form"],
                "minutes": p["minutes"],
                "selected_by_percent": p["selected_by_percent"],
            }
            for p in data.get("elements", [])[:10]
        ],
    ).model_dump()


def get_fixtures() -> list[dict[str, Any]]:
    """Return a trimmed list of upcoming or recent FPL fixtures."""
    fixtures = fpl_service.get_fixtures()
    return fixtures[:20]


def get_player_news() -> list[dict[str, Any]]:
    """Return player availability and injury news from FPL bootstrap data."""
    return fpl_service.get_player_news()


def get_live_gameweek_data(gameweek: int) -> dict[str, Any]:
    """Return live FPL data for a specific gameweek.

    Args:
        gameweek: The FPL gameweek number to fetch.
    """
    return fpl_service.get_event_live(gameweek)


def get_player_detail(player_id: int) -> dict[str, Any]:
    """Return detailed FPL history and fixture data for one player.

    Args:
        player_id: The FPL player ID.
    """
    return fpl_service.get_element_summary(player_id)


from __future__ import annotations

from typing import Any


def get_manager_team(team_id: int, gameweek: int) -> dict[str, Any]:
    """Return a manager's entry details and picks for a given gameweek."""
    entry = fpl_service.get_entry(team_id)
    picks = fpl_service.get_entry_picks(team_id, gameweek)
    return {
        "entry": entry,
        "picks": picks,
    }


def get_manager_team_summary(team_id: int, gameweek: int) -> dict[str, Any]:
    """
    Return starters, bench, and automatic substitutions for a manager.

    Also attaches player names and gameweek points.
    """
    team_data = get_manager_team(team_id, gameweek)
    bootstrap = fpl_service.get_bootstrap_static()
    live = fpl_service.get_event_live(gameweek)

    picks_data = team_data["picks"]

    # player id -> player full name
    player_names: dict[int, str] = {
        player["id"]: f"{player['first_name']} {player['second_name']}"
        for player in bootstrap["elements"]
    }

    # player id -> total points in that gameweek
    player_points: dict[int, int] = {
        item["id"]: item["stats"]["total_points"]
        for item in live["elements"]
    }

    starters = []
    bench = []

    for pick in picks_data["picks"]:
        player_id = pick["element"]
        player_info = {
            "element": player_id,
            "name": player_names.get(player_id, f"Player {player_id}"),
            "position": pick["position"],
            "points": player_points.get(player_id, 0),
            "multiplier": pick["multiplier"],
            "is_captain": pick["is_captain"],
            "is_vice_captain": pick["is_vice_captain"],
        }

        if pick["position"] <= 11:
            starters.append(player_info)
        else:
            bench.append(player_info)

    automatic_substitutions = []
    for sub in picks_data.get("automatic_subs", []):
        out_id = sub["element_out"]
        in_id = sub["element_in"]

        automatic_substitutions.append(
            {
                "out_element": out_id,
                "out_name": player_names.get(out_id, f"Player {out_id}"),
                "out_points": player_points.get(out_id, 0),
                "in_element": in_id,
                "in_name": player_names.get(in_id, f"Player {in_id}"),
                "in_points": player_points.get(in_id, 0),
            }
        )

    return {
        "entry": team_data["entry"],
        "gameweek": gameweek,
        "starters": starters,
        "bench": bench,
        "automatic_substitutions": automatic_substitutions,
    }


In [52]:
get_manager_team(5196911, 30)["picks"]

{'active_chip': None,
 'automatic_subs': [],
 'entry_history': {'event': 30,
  'points': 56,
  'total_points': 1637,
  'rank': 1627681,
  'rank_sort': 1691629,
  'overall_rank': 2205318,
  'percentile_rank': 15,
  'bank': 21,
  'value': 1021,
  'event_transfers': 0,
  'event_transfers_cost': 0,
  'points_on_bench': 8},
 'picks': [{'element': 470,
   'position': 1,
   'multiplier': 1,
   'is_captain': False,
   'is_vice_captain': False,
   'element_type': 1},
  {'element': 8,
   'position': 2,
   'multiplier': 1,
   'is_captain': False,
   'is_vice_captain': False,
   'element_type': 2},
  {'element': 260,
   'position': 3,
   'multiplier': 1,
   'is_captain': False,
   'is_vice_captain': False,
   'element_type': 2},
  {'element': 373,
   'position': 4,
   'multiplier': 1,
   'is_captain': False,
   'is_vice_captain': False,
   'element_type': 2},
  {'element': 74,
   'position': 5,
   'multiplier': 1,
   'is_captain': False,
   'is_vice_captain': False,
   'element_type': 2},
  {'elem

In [66]:
summary = get_manager_team_summary(team_id=5196911, gameweek=30)

print("STARTERS")
for player in summary["starters"]:
    print(f"{player['name']} - {player['points']} pts")

print("\nBENCH")
for player in summary["bench"]:
    print(f"{player['name']} - {player['points']} pts")

print("\nAUTO SUBS")
for sub in summary["automatic_substitutions"]:
    print(f"{sub['out_name']} -> {sub['in_name']}")

STARTERS
Martin Dúbravka - 6 pts
Jurriën Timber - 1 pts
Marc Guéhi - 2 pts
Virgil van Dijk - 2 pts
Adrien Truffert - 11 pts
Declan Rice - 3 pts
Bruno Borges Fernandes - 10 pts
Antoine Semenyo - 2 pts
João Pedro Junqueira de Jesus - 2 pts
Igor Thiago Nascimento Rodrigues - 6 pts
Hugo Ekitiké - 1 pts

BENCH
Robert Lynch Sánchez - 3 pts
Harry Wilson - 3 pts
Morgan Rogers - 2 pts
Ibrahima Konaté - 0 pts

AUTO SUBS


In [54]:
get_player_detail(470)

{'fixtures': [{'id': 313,
   'code': 2562207,
   'team_h': 3,
   'team_h_score': None,
   'team_a': 6,
   'team_a_score': None,
   'event': 32,
   'finished': False,
   'minutes': 0,
   'provisional_start_time': False,
   'kickoff_time': '2026-04-11T14:00:00Z',
   'event_name': 'Gameweek 32',
   'is_home': True,
   'difficulty': 3},
  {'id': 329,
   'code': 2562223,
   'team_h': 16,
   'team_h_score': None,
   'team_a': 3,
   'team_a_score': None,
   'event': 33,
   'finished': False,
   'minutes': 0,
   'provisional_start_time': False,
   'kickoff_time': '2026-04-19T13:00:00Z',
   'event_name': 'Gameweek 33',
   'is_home': False,
   'difficulty': 3},
  {'id': 334,
   'code': 2562228,
   'team_h': 3,
   'team_h_score': None,
   'team_a': 13,
   'team_a_score': None,
   'event': 34,
   'finished': False,
   'minutes': 0,
   'provisional_start_time': False,
   'kickoff_time': '2026-04-26T13:00:00Z',
   'event_name': 'Gameweek 34',
   'is_home': True,
   'difficulty': 4},
  {'id': 347,
  

In [55]:
get_bootstrap_data()

{'total_players': 825,
 'total_teams': 20,
 'total_events': 38,
 'sample_players': [{'id': 1,
   'web_name': 'Raya',
   'team': 1,
   'now_cost': 60,
   'total_points': 129,
   'form': '7.0',
   'minutes': 2790,
   'selected_by_percent': '34.6'},
  {'id': 2,
   'web_name': 'Arrizabalaga',
   'team': 1,
   'now_cost': 40,
   'total_points': 0,
   'form': '0.0',
   'minutes': 0,
   'selected_by_percent': '0.4'},
  {'id': 3,
   'web_name': 'Hein',
   'team': 1,
   'now_cost': 40,
   'total_points': 0,
   'form': '0.0',
   'minutes': 0,
   'selected_by_percent': '0.2'},
  {'id': 4,
   'web_name': 'Setford',
   'team': 1,
   'now_cost': 39,
   'total_points': 0,
   'form': '0.0',
   'minutes': 0,
   'selected_by_percent': '0.2'},
  {'id': 5,
   'web_name': 'Gabriel',
   'team': 1,
   'now_cost': 72,
   'total_points': 173,
   'form': '10.0',
   'minutes': 2165,
   'selected_by_percent': '43.0'},
  {'id': 6,
   'web_name': 'Saliba',
   'team': 1,
   'now_cost': 61,
   'total_points': 107,
  

In [23]:
get_player_news()[30:40]

[{'player_id': 61,
  'name': 'Dendoncker',
  'team_id': 2,
  'team_name': 'Aston Villa',
  'status': 'u',
  'chance_of_playing_next_round': 0,
  'chance_of_playing_this_round': 0,
  'news': 'Has joined Real Oviedo permanently.'},
 {'player_id': 62,
  'name': 'Jimoh-Aloba',
  'team_id': 2,
  'team_name': 'Aston Villa',
  'status': 'u',
  'chance_of_playing_next_round': 0,
  'chance_of_playing_this_round': 0,
  'news': 'has joined West Brom on loan for the rest of the season.'},
 {'player_id': 63,
  'name': 'Young',
  'team_id': 2,
  'team_name': 'Aston Villa',
  'status': 'u',
  'chance_of_playing_next_round': 0,
  'chance_of_playing_this_round': 0,
  'news': 'has joined Reading on loan for the rest of the season.'},
 {'player_id': 65,
  'name': 'Redmond',
  'team_id': 2,
  'team_name': 'Aston Villa',
  'status': 'u',
  'chance_of_playing_next_round': 0,
  'chance_of_playing_this_round': 0,
  'news': 'has joined Huddersfield Town on loan for the rest of the season.'},
 {'player_id': 456

## Adding FPL analysis logic

In [56]:
# src/tools/fpl_analysis.py
from __future__ import annotations

from typing import Any

from agents import function_tool

#from src.services.fpl_service import FPLService

fpl_service = FPLService()


def rank_captain_candidates(top_n: int = 5) -> list[dict[str, Any]]:
    data = fpl_service.get_bootstrap_static()
    players = data.get("elements", [])

    scored = []
    for p in players:
        minutes = p.get("minutes", 0)
        form = float(p.get("form", 0) or 0)
        points_per_game = float(p.get("points_per_game", 0) or 0)
        ownership = float(p.get("selected_by_percent", 0) or 0)

        if minutes < 400:
            continue

        score = (
            form * 0.45
            + points_per_game * 0.35
            + min(ownership, 50) * 0.02
        )

        scored.append(
            {
                "id": p["id"],
                "name": p["web_name"],
                "team_id": p["team"],
                "position_id": p["element_type"],
                "price": p["now_cost"] / 10,
                "form": form,
                "points_per_game": points_per_game,
                "ownership": ownership,
                "score": round(score, 2),
            }
        )

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_n]

In [57]:
rank_captain_candidates()

[{'id': 449,
  'name': 'B.Fernandes',
  'team_id': 14,
  'position_id': 3,
  'price': 10.3,
  'form': 10.3,
  'points_per_game': 6.8,
  'ownership': 44.5,
  'score': 7.91},
 {'id': 5,
  'name': 'Gabriel',
  'team_id': 1,
  'position_id': 2,
  'price': 7.2,
  'form': 10.0,
  'points_per_game': 6.9,
  'ownership': 43.0,
  'score': 7.78},
 {'id': 249,
  'name': 'João Pedro',
  'team_id': 7,
  'position_id': 4,
  'price': 7.8,
  'form': 7.7,
  'points_per_game': 5.3,
  'ownership': 50.4,
  'score': 6.32},
 {'id': 267,
  'name': 'Sarr',
  'team_id': 8,
  'position_id': 3,
  'price': 6.3,
  'form': 9.0,
  'points_per_game': 4.7,
  'ownership': 3.8,
  'score': 5.77},
 {'id': 485,
  'name': 'Gordon',
  'team_id': 15,
  'position_id': 3,
  'price': 7.4,
  'form': 9.3,
  'points_per_game': 3.9,
  'ownership': 7.4,
  'score': 5.7}]

## Creating an agent to make a decision

In [58]:
# src/agents/fpl_advisor.py
from agents import Agent

#from src.tools.fpl_api import (
 #   get_bootstrap_data,
  #  get_fixtures,
 #   get_live_gameweek_data,
   # get_player_detail,
  #  get_manager_team,
# )
# from src.tools.fpl_analysis import rank_captain_candidates


fpl_advisor = Agent(
    name="FPL Advisor",
    instructions=(
        "You are an expert Fantasy Premier League assistant. "
        "Use the available tools to answer with practical, evidence-based FPL advice. "
        "Do not invent injuries, fixtures, or stats. "
        "When recommending transfers or captaincy, explain the reasoning clearly, "
        "including form, minutes, fixture quality, and risk."
    ),
    tools=[
        function_tool(get_bootstrap_data),
        function_tool(get_fixtures),
        function_tool(get_live_gameweek_data),
        function_tool(get_player_detail),
        function_tool(get_manager_team),
        function_tool(rank_captain_candidates),
    ],
)

In [59]:
result = await Runner.run(
        fpl_advisor,
        "Rank the best captain picks for this gameweek and explain the top 3."
    )
print(result.final_output)

Here are the best captain picks for this gameweek, ranked:

1. Bruno Fernandes (Man United, MID)
2. Gabriel (Arsenal, DEF)
3. João Pedro (Brighton, FWD)
4. Sarr (Wolves, MID)
5. Gordon (Newcastle, MID)

Explanations for the top 3:

1. Bruno Fernandes
   - Form: 10.3 (highest in the league right now).
   - Points per game: 6.8.
   - Reasoning: Bruno is central to all of Manchester United’s attacks—he's on penalties, created a lot of chances lately, and is a reliable performer with both assist and goal potential. His consistency and involvement in big chances make him a top captaincy pick.

2. Gabriel (Arsenal)
   - Form: 10.0.
   - Points per game: 6.9.
   - Reasoning: As a defender, Gabriel offers huge goal threat from set pieces, and Arsenal’s solid defense means a clean sheet is likely. He’s also in great scoring form, making him one of the rare defenders worth captaining.

3. João Pedro (Brighton)
   - Form: 7.7.
   - Points per game: 5.3.
   - Reasoning: João Pedro is Brighton’s mo

In [ ]:
fpl_advisor = Agent(
    name="FPL Advisor",
    instructions=(
        "You are an expert Fantasy Premier League assistant. "
        "Use the available tools to answer with practical, evidence-based FPL advice. "
        "Do not invent injuries, fixtures, or stats. "
        "When recommending transfers or captaincy, explain the reasoning clearly, "
        "including form, minutes, fixture quality, and risk."
    ),
    tools=[
        function_tool(get_bootstrap_data),
        function_tool(get_fixtures),
        function_tool(get_live_gameweek_data),
        function_tool(get_player_detail),
        function_tool(get_manager_team),
        function_tool(rank_captain_candidates),
    ],
)

## Adding an agent that returns the manager's team

In [5]:
fpl_team_agent = Agent(
    name="FPL Team Viewer",
    instructions=(
        "You only retrieve a manager's FPL team for a specific gameweek. "
        "Use the `get_manager_team` tool when the user provides a team ID and gameweek. "
        "Do not answer captaincy, transfer, fixture, or general analysis questions. "
        "If either the team ID or gameweek is missing, ask for the missing value."
    ),
    tools=[function_tool(get_manager_team)],
)


In [9]:
TEAM_PROMPT_TEMPLATE = (
    "Use get_manager_team with team_id={team_id} and gameweek={gameweek}. "
    "Return only the manager team data."
)

team_id = 15638
gameweek = 31

result = await Runner.run(
    fpl_team_agent,
    TEAM_PROMPT_TEMPLATE.format(team_id=team_id, gameweek=gameweek),
)

print(result.final_output)


{
  "entry": {
    "id": 15638,
    "name": "FPL MasterChef",
    "player_first_name": "Tshepo",
    "player_last_name": "Vilakazi",
    "player_region_name": "South Africa",
    "summary_overall_points": 1739,
    "summary_overall_rank": 1143693,
    "summary_event_points": 56,
    "summary_event_rank": 2196409,
    "current_event": 31,
    "leagues": {...}
  },
  "picks": {
    "active_chip": null,
    "automatic_subs": [],
    "entry_history": {
      "event": 31,
      "points": 56,
      "total_points": 1739,
      "rank": 2196409,
      "overall_rank": 1143693,
      "percentile_rank": 20,
      "bank": 5,
      "value": 1020,
      "event_transfers": 0,
      "event_transfers_cost": 0,
      "points_on_bench": 0
    },
    "picks": [
      {"element": 470, "position": 1, "multiplier": 1, "element_type": 1},
      {"element": 683, "position": 2, "multiplier": 1, "element_type": 2},
      {"element": 260, "position": 3, "multiplier": 1, "element_type": 2},
      {"element": 348, "

## Experimenting with structured Output

In [68]:
TEAM_PROMPT_TEMPLATE = (
    "Use get_manager_team with team_id={team_id} and gameweek={gameweek}. "
    "Return only the manager team data in the required schema."
)

team_id = 5196911
gameweek = 30

result = await Runner.run(
    fpl_team_agent,
    TEAM_PROMPT_TEMPLATE.format(team_id=team_id, gameweek=gameweek),
)

team_doc: TeamDoc = result.final_output
print(team_doc)

team_id=5196911 gameweek=30 manager_name='Tumelo Konaite' team_name='Captain Levi' entry=EntryInfo(id=5196911, player_first_name='Tumelo', player_last_name='Konaite', player_region_name='South Africa', name='Captain Levi', summary_overall_points=1697, summary_event_points=60, summary_overall_rank=1955689, last_deadline_bank=62, last_deadline_value=1021, last_deadline_total_transfers=30) active_chip=None picks=[PickItem(element=470, position=1, multiplier=1, is_captain=False, is_vice_captain=False, element_type=1), PickItem(element=8, position=2, multiplier=1, is_captain=False, is_vice_captain=False, element_type=2), PickItem(element=260, position=3, multiplier=1, is_captain=False, is_vice_captain=False, element_type=2), PickItem(element=373, position=4, multiplier=1, is_captain=False, is_vice_captain=False, element_type=2), PickItem(element=74, position=5, multiplier=1, is_captain=False, is_vice_captain=False, element_type=2), PickItem(element=21, position=6, multiplier=1, is_captain=F

In [69]:
# Saving it as JSON
def save_team_as_json(team_doc: TeamDoc, output_path: str = "team_doc.json") -> None:
    Path(output_path).write_text(
        team_doc.model_dump_json(indent=2),
        encoding="utf-8",
    )

In [70]:
save_team_as_json(team_doc, "fpl_team_gw30.json")